# 02 -- model training

trains a small neural network on the board state data generated by
notebook 01. the model learns to predict how much score remains
from any given board position.

architecture: 16 -> dense 128 relu -> dense 64 relu -> 1 linear.

exports to tensorflow.js format (model.json + weights.bin).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'tensorflow version: {tf.__version__}')

## load and prepare data

In [ ]:
data = np.load('training_data.npz')
boards = data['boards']
scores = data['scores']

print(f'boards: {boards.shape}')
print(f'scores: {scores.shape}')
print(f'score range: {scores.min():.0f} - {scores.max():.0f}')

# normalize scores to [0, 1] range for training stability
score_max = scores.max()
scores_norm = scores / score_max

# shuffle and split: 90% train, 10% validation
n = len(boards)
indices = np.random.permutation(n)
split = int(n * 0.9)

x_train = boards[indices[:split]]
y_train = scores_norm[indices[:split]]
x_val = boards[indices[split:]]
y_val = scores_norm[indices[split:]]

print(f'\ntrain: {x_train.shape[0]} samples')
print(f'val:   {x_val.shape[0]} samples')

## build model

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(16,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

## train

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=256,
    verbose=1
)

## evaluate

In [ ]:
val_loss, val_mae = model.evaluate(x_val, y_val, verbose=0)

# convert mae back to actual score units
val_mae_score = val_mae * score_max

print(f'validation loss (mse): {val_loss:.6f}')
print(f'validation mae (normalized): {val_mae:.4f}')
print(f'validation mae (score units): {val_mae_score:.0f}')
print(f'\nfor reference, score_max = {score_max:.0f}')

# spot check predictions vs actuals
preds = model.predict(x_val[:10], verbose=0).flatten()
actuals = y_val[:10]
print(f'\nspot check (normalized):')
for i in range(10):
    print(f'  predicted: {preds[i]:.4f}  actual: {actuals[i]:.4f}  '
          f'error: {abs(preds[i]-actuals[i])*score_max:.0f}')

## save keras model

In [ ]:
model.save('board_evaluator.keras')
print('saved board_evaluator.keras')

# also save score_max so js can denormalize predictions
np.save('score_max.npy', np.array([score_max], dtype=np.float32))
print(f'saved score_max.npy (value: {score_max:.0f})')

## export to tensorflow.js

In [ ]:
!pip install tensorflowjs -q

In [ ]:
import tensorflowjs as tfjs

tfjs.converters.save_keras_model(model, 'tfjs_model')
print('exported to tfjs_model/')

import os
for f in os.listdir('tfjs_model'):
    size = os.path.getsize(os.path.join('tfjs_model', f))
    print(f'  {f} ({size} bytes)')

## package output

In [ ]:
import shutil, os

os.makedirs('output', exist_ok=True)

# copy tfjs model files
for f in os.listdir('tfjs_model'):
    shutil.copy(os.path.join('tfjs_model', f), os.path.join('output', f))

# copy score_max
shutil.copy('score_max.npy', 'output/score_max.npy')

shutil.make_archive('02_model_training_output', 'zip', 'output')
print('done. download 02_model_training_output.zip')
print('\ncontents:')
for f in os.listdir('output'):
    print(f'  {f}')